# Quantum random numbers

A qubit in the state `|+⟩` (after a Hadamard gate) gives `0` or `1` with equal probability when measured. Measuring `n` such qubits gives a random `n`-bit number, uniformly distributed between `0` and `2ⁿ − 1`.

This example generates random 5-bit numbers (0–31) on a QaaS backend.

In [ ]:
%pip install pylatexenc
%pip install matplotlib
%pip install 'qaas[iqm_qiskit]>=1.1.0' --extra-index-url https://opencode.it4i.eu/api/v4/projects/107/packages/pypi/simple

In [ ]:
import os

os.environ["QPROVIDER_LOGLEVEL"] = "INFO"

# Connect to system

In [ ]:
from qheappe_client import QBackend, QJob, QProvider

lexis_project = "EU-xx-xxx"
resource_name = "VLQ-EU"

provider = QProvider(lexis_project)

## or define manually LEXIS access token e.g. via dotenv

## %pip install python-dotenv
## from dotenv import load_dotenv
## load_dotenv()

## provider = QProvider(LEXIS_PROJECT, token)

In [ ]:
backend: QBackend = provider.get_backend(resource_name, backend_name="VLQ")

print("Qubits:", len(backend.architecture.qubits))
print("Native gates:", list(backend.architecture.gates.keys()))

# Define the circuit

In [ ]:
from qiskit import QuantumCircuit

N_BITS = 5  # random numbers from 0 to 2**N_BITS - 1

qc = QuantumCircuit(N_BITS, name="random_numbers")
qc.h(range(N_BITS))
qc.measure_all()

qc.draw()

# Transpile the circuit

`backend.transpile()` maps the circuit to the backend's native gates and qubits.

In [ ]:
qc_transpiled = backend.transpile(qc)

# Run the circuit

`memory=True` keeps the result of every shot, which is the sequence of random numbers. `wait_for_final_state()` follows HEAppE's event stream, so it returns as soon as the job finishes. It's more efficient than calling `job.status()` in a loop.

In [ ]:
from time import time

SHOTS = 1000

start = time()
job: QJob = backend.run(qc_transpiled, shots=SHOTS, memory=True, job_name="random_numbers")
print("HEAppE job ID:", job.job_id())

job.wait_for_final_state(timeout=600)
result = job.result()
print(f"Finished in {time() - start:.1f} s with status {job.status().name}")

# Random numbers

Each per-shot bitstring is one random number. Bitstrings follow the Qiskit order (qubit 0 is the rightmost bit), so `int(bits, 2)` gives the number.

In [ ]:
numbers = [int(bits, 2) for bits in result.get_memory()]
print("First 20 random numbers:", numbers[:20])

# How uniform are they?

On an ideal device, every number from 0 to 31 appears with probability 1/32. Hardware noise makes some numbers a bit more or less frequent.

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt

n_values = 2**N_BITS
frequency = Counter(numbers)
expected = SHOTS / n_values

plt.figure(figsize=(12, 4))
plt.bar(range(n_values), [frequency.get(v, 0) for v in range(n_values)])
plt.axhline(expected, color="red", linestyle="--", label=f"ideal ({expected:.1f})")
plt.xlabel("Random number")
plt.ylabel("Count")
plt.legend()
plt.show()

counts = [frequency.get(v, 0) for v in range(n_values)]
print(f"Least frequent: {min(counts)}, most frequent: {max(counts)}, ideal: {expected:.1f}")

In [ ]:
# Fraction of 1s per qubit; ideally 0.5 each
for bit in range(N_BITS):
    ones = sum((n >> bit) & 1 for n in numbers) / SHOTS
    print(f"Qubit {bit}: P(1) = {ones:.3f}")